In [1]:
import requests
import requests, time

In [2]:
# Extrair mercado

BASE = "https://api.coingecko.com/api/v3"

def extrair_mercado(vs="usd", total=250, por_pagina=250, tentativas=3):
    """Extrai o snapshot de mercado das top N moedas, paginando."""
    todas, pagina = [], 1
    while len(todas) < total:
        params = {"vs_currency": vs, "order": "market_cap_desc",
                  "per_page": por_pagina, "page": pagina}

        for t in range(tentativas):
            r = requests.get(f"{BASE}/coins/markets", params=params, timeout=30)
            if r.status_code == 429:                 # rate limit
                time.sleep(2 ** t)                   # backoff exponencial
                continue
            r.raise_for_status()
            lote = r.json()
            break
        else:
            # só chega aqui se o for terminou SEM break → todas as tentativas foram 429
            raise RuntimeError(
                f"Rate limit persistente na página {pagina} após {tentativas} tentativas"
            )

        if not lote:
            print("Não existem mais dados para coletar. Parando.")
            break
        todas.extend(lote)
        pagina += 1
    return todas[:total]

extracao = extrair_mercado()

for moeda in extracao[:5]:
    print(moeda['id'])

bitcoin
ethereum
tether
binancecoin
usd-coin


In [3]:
extracao

[{'id': 'bitcoin',
  'symbol': 'btc',
  'name': 'Bitcoin',
  'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400',
  'current_price': 64497,
  'market_cap': 1294573351566,
  'market_cap_rank': 1,
  'fully_diluted_valuation': 1294575157957,
  'total_volume': 20433482485,
  'high_24h': 64933,
  'low_24h': 64114,
  'price_change_24h': -219.99028005944274,
  'price_change_percentage_24h': 0.5,
  'market_cap_change_24h': -4041100704.5566406,
  'market_cap_change_percentage_24h': -0.31119,
  'circulating_supply': 20066568.0,
  'total_supply': 20066612.0,
  'max_supply': 21000000.0,
  'ath': 126080,
  'ath_change_percentage': -48.84464,
  'ath_date': '2025-10-06T10:57:42.000Z',
  'atl': 67.81,
  'atl_change_percentage': 95015.13794,
  'atl_date': '2013-07-05T16:00:00.000Z',
  'roi': None,
  'last_updated': '2026-08-06T17:41:30.000Z'},
 {'id': 'ethereum',
  'symbol': 'eth',
  'name': 'Ethereum',
  'image': 'https://coin-images.coingecko.com/coins/images/279/

In [4]:
import psycopg2
from datetime import datetime, timezone
# from extrair import extrair_mercado

In [5]:
# Transformar os dados

def transformar(bruto):
    """Seleciona e limpa apenas os campos que interessam."""
    agora = datetime.now(timezone.utc)
    return [(
        m["id"], m["symbol"], m["name"],
        m["current_price"], m["total_volume"],
        m["market_cap"], m.get("price_change_percentage_24h"), agora,
    ) for m in bruto]


dados_transformados = transformar(extracao)
dados_transformados

[('bitcoin',
  'btc',
  'Bitcoin',
  64497,
  20433482485,
  1294573351566,
  0.5,
  datetime.datetime(2026, 8, 6, 17, 44, 31, 234988, tzinfo=datetime.timezone.utc)),
 ('ethereum',
  'eth',
  'Ethereum',
  1909.17,
  8060548974,
  230549129440,
  2.1,
  datetime.datetime(2026, 8, 6, 17, 44, 31, 234988, tzinfo=datetime.timezone.utc)),
 ('tether',
  'usdt',
  'Tether',
  0.999218,
  34702441841,
  183408017649,
  0.0,
  datetime.datetime(2026, 8, 6, 17, 44, 31, 234988, tzinfo=datetime.timezone.utc)),
 ('binancecoin',
  'bnb',
  'BNB',
  592.15,
  552774524,
  78864995066,
  -1.3,
  datetime.datetime(2026, 8, 6, 17, 44, 31, 234988, tzinfo=datetime.timezone.utc)),
 ('usd-coin',
  'usdc',
  'USDC',
  0.999605,
  9579176252,
  71881136828,
  0.0,
  datetime.datetime(2026, 8, 6, 17, 44, 31, 234988, tzinfo=datetime.timezone.utc)),
 ('ripple',
  'xrp',
  'XRP',
  1.041,
  1305258967,
  65069678198,
  -1.5,
  datetime.datetime(2026, 8, 6, 17, 44, 31, 234988, tzinfo=datetime.timezone.utc)),
 ('so

In [6]:
# Importando dados para o banco

def carregar(linhas):
    """Grava no Postgres com UPSERT (idempotente na mesma coleta)."""
    conn = psycopg2.connect(
        host="localhost", port=5433, dbname="criptoflow",
        user="criptoflow", password="criptoflow",
    )
    with conn, conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO mercado_bruto
            (id, simbolo, nome, preco_usd, volume_24h,
             market_cap, variacao_24h, coletado_em)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (id, coletado_em) DO NOTHING
        """, linhas)
    conn.close()


carregar(dados_transformados)

In [7]:

## Carregar Moedas

def carregar_moedas(bruto):
    linhas = [(m['id'], m['symbol'], m['name']) for m in bruto]

    conn = psycopg2.connect(
        host="localhost", port=5433, dbname="criptoflow",
        user="criptoflow", password="criptoflow",
    )

    with conn, conn.cursor() as cur:
            cur.executemany("""
                INSERT INTO moedas
                (id, simbolo, nome)
                VALUES (%s,%s,%s)
                ON CONFLICT (id) DO NOTHING
            """, linhas)
    conn.close()


carregar_moedas(extracao)



In [9]:
linhas = [(m['id'], m['symbol'], m['name']) for m in extracao]
linhas

conn = psycopg2.connect(
        host="localhost", port=5433, dbname="criptoflow",
        user="criptoflow", password="criptoflow",
    )

with conn, conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO mercado_bruto
            (id, simbolo, nome, preco_usd, volume_24h,
             market_cap, variacao_24h, coletado_em)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (id, coletado_em) DO NOTHING
        """, linhas)
    conn.close()

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 17)

In [ ]:
from datetime import datetime, timezone

agora = datetime.now().strftime("%Y-%m-%d %H:%M")
agora


'2026-07-29 15:14'

datetime.datetime(2026, 7, 26, 14, 26, 16, 121179)

In [10]:
import io, json
from datetime import datetime, timezone
import boto3
import pandas as pd
from extrair import extrair_mercado
from dotenv import load_dotenv
import os

# --- Config do MinIO (porta 9100 = API S3, a que remapeamos) ---
load_dotenv()

MINIO_ENDPOINT = "http://localhost:9100"
MINIO_KEY      = os.getenv('MINIO_KEY')
MINIO_SECRET   = os.getenv('MINIO_SECRET')
BUCKET         = "criptoflow"


def cliente_s3():
    return boto3.client(
        "s3",
        endpoint_url=MINIO_ENDPOINT,
        aws_access_key_id=MINIO_KEY,
        aws_secret_access_key=MINIO_SECRET,
    )

def garantir_bucket(s3):
    existentes = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    if BUCKET not in existentes:
        s3.create_bucket(Bucket=BUCKET)
        print(f"Bucket '{BUCKET}' criado.")
    else:
        print('Listando Buckets:')
        for bucket in existentes:
            print(bucket)

def gravar_bronze(bruto, coletado_em):
    df = pd.DataFrame(bruto)              # o JSON cru vira DataFrame (TODAS as colunas)
    df["coletado_em"] = coletado_em      # metadado da coleta

    # Parquet é colunar e não gosta de células aninhadas (dict/list).
    # Serializamos essas colunas como texto JSON para caberem no formato.
    for col in df.columns:
        if df[col].apply(lambda v: isinstance(v, (dict, list))).any():
            df[col] = df[col].apply(lambda v: json.dumps(v) if isinstance(v, (dict, list)) else v)

    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False)   # DataFrame -> Parquet, em memória
    buffer.seek(0)

    dia = coletado_em.strftime("%Y-%m-%d")
    ts  = coletado_em.strftime("%Y%m%dT%H%M%S")
    chave = f"bronze/mercado/dia={dia}/mercado_{ts}.parquet"

    s3 = cliente_s3()
    garantir_bucket(s3)
    s3.put_object(Bucket=BUCKET, Key=chave, Body=buffer.getvalue())
    print(f"Gravado s3://{BUCKET}/{chave} ({len(df)} linhas, {len(df.columns)} colunas)")


Listando Buckets:
criptoflow


## Silver

In [12]:
import io, json
from datetime import datetime, timezone
import boto3
import pandas as pd
from extrair import extrair_mercado
from dotenv import load_dotenv
import os

# --- Config do MinIO (porta 9100 = API S3, a que remapeamos) ---
load_dotenv()

MINIO_ENDPOINT = "http://localhost:9100"
MINIO_KEY      = os.getenv('MINIO_KEY')
MINIO_SECRET   = os.getenv('MINIO_SECRET')
BUCKET         = "criptoflow"

# Mapa: coluna da bronze (fonte) -> coluna padronizada da silver
COLUNAS = {
    "id": "id", "symbol": "simbolo", "name": "nome",
    "current_price": "preco_usd", "total_volume": "volume_24h",
    "market_cap": "market_cap", "market_cap_rank": "rank",
    "price_change_percentage_24h": "variacao_24h", "coletado_em": "coletado_em",
}


def cliente_s3():
    return boto3.client("s3", 
                        endpoint_url=MINIO_ENDPOINT,
                        aws_access_key_id=MINIO_KEY, 
                        aws_secret_access_key=MINIO_SECRET)

def ler_bronze(s3):
    """Fan-in: lê TODOS os Parquet da bronze e junta num só DataFrame."""
    resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="bronze/mercado/")
    dfs = []
    for obj in resp.get("Contents", []):
        if obj["Key"].endswith(".parquet"):
            corpo = s3.get_object(Bucket=BUCKET, Key=obj["Key"])["Body"].read()
            dfs.append(pd.read_parquet(io.BytesIO(corpo)))
    if not dfs:
        raise RuntimeError("Nenhum arquivo encontrado na bronze.")
    return pd.concat(dfs, ignore_index=True)

def transformar_silver(df):
    # 1. seleciona e renomeia (padroniza o schema)
    df = df[list(COLUNAS.keys())].rename(columns=COLUNAS)
    # 2. tipa: números viram números; texto ruim vira NaN em vez de quebrar
    for c in ["preco_usd", "volume_24h", "market_cap", "rank", "variacao_24h"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["coletado_em"] = pd.to_datetime(df["coletado_em"], utc=True)
    # 3. deduplica: uma linha por (id, coletado_em), mantendo a mais recente
    df = df.sort_values("coletado_em").drop_duplicates(subset=["id", "coletado_em"], keep="last")
    return df

def gravar_silver(df):
    s3 = cliente_s3()
    df = df.copy()
    df["dia"] = df["coletado_em"].dt.strftime("%Y-%m-%d")
    for dia, grupo in df.groupby("dia"):
        grupo = grupo.drop(columns=["dia"])
        buffer = io.BytesIO()
        grupo.to_parquet(buffer, index=False)
        chave = f"silver/mercado/dia={dia}/mercado.parquet"   # nome fixo -> overwrite idempotente
        s3.put_object(Bucket=BUCKET, Key=chave, Body=buffer.getvalue())
        print(f"silver: s3://{BUCKET}/{chave} ({len(grupo)} linhas)")

In [13]:
s3 = cliente_s3()

In [ ]:
df = ler_bronze(s3)

(250, 27)
['id', 'symbol', 'name', 'image', 'current_price', 'market_cap', 'market_cap_rank', 'fully_diluted_valuation', 'total_volume', 'high_24h', 'low_24h', 'price_change_24h', 'price_change_percentage_24h', 'market_cap_change_24h', 'market_cap_change_percentage_24h', 'circulating_supply', 'total_supply', 'max_supply', 'ath', 'ath_change_percentage', 'ath_date', 'atl', 'atl_change_percentage', 'atl_date', 'roi', 'last_updated', 'coletado_em']


colunas faltando: set()


In [15]:
silver = transformar_silver(df)

## Gold

In [30]:
from gold import ler_silver, cliente_s3, construir_dim_moeda, construir_fct_precos

In [20]:
s3 = cliente_s3()

In [23]:
silver = ler_silver(s3)

In [26]:
dim = silver[['id', 'nome', 'simbolo']]
dim.shape

(250, 3)

In [28]:
dim.drop_duplicates(subset=['id']).sort_values('id').reset_index(drop=True)

,id,nome,simbolo
0,1inch,1INCH,1inch
1,a7a5,A7A5,a7a5
2,aave,Aave,aave
3,aerodrome-finance,Aerodrome Finance,aero
4,agora-dollar,AUSD,ausd
...,...,...,...
245,zama,Zama,zama
246,zano,Zano,zano
247,zcash,Zcash,zec
248,zebec-network,Zebec Network,zbcn


In [29]:
dim.insert(0,'moeda_sk', range(1, len(dim) +1))
print(dim.shape)
print(dim['id'].is_unique)
print(dim.head())

(250, 4)
True
   moeda_sk         id           nome simbolo
0         1    bitcoin        Bitcoin     btc
1         2  jasmycoin      JasmyCoin   jasmy
2         3     plasma         Plasma     xpl
3         4     crvusd         crvUSD  crvusd
4         5      syrup  Maple Finance   syrup


In [32]:
fct = construir_fct_precos(silver, dim)
print(fct['moeda_sk'].isna().sum())

0
